# 06_retrieval_evaluation.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook evalúa los resultados de recuperación semántica generados en el Notebook 05.

## Objetivo
Comparar MiniLM y BGE-M3 usando evaluación manual de relevancia sobre los resultados Top-K.

## Entradas
- `retrieval_manual_evaluation_template.xlsx`
- Opcional: `retrieval_results_all_models.csv`

## Salidas
- `retrieval_manual_evaluation_completed.xlsx`
- `retrieval_metrics_by_model.csv`
- `retrieval_metrics_by_question.csv`
- `retrieval_metrics_by_category.csv`

## Nota metodológica
La similitud coseno no demuestra por sí sola relevancia. Por eso este notebook permite marcar manualmente si cada resultado recuperado es relevante (`1`) o no relevante (`0`) y calcular métricas como Precision@1, Precision@3 y Precision@5.


In [ ]:
# =====================================================
# 06_retrieval_evaluation.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas openpyxl matplotlib

## 1. Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files

pd.set_option('display.max_colwidth', 160)

print('Librerías cargadas correctamente')

## 2. Cargar archivo de evaluación

Sube el archivo generado en el Notebook 05:

`retrieval_manual_evaluation_template.xlsx`

Este archivo contiene los resultados Top-5 de MiniLM y BGE-M3 para cada pregunta del gold standard.

In [ ]:
print('Sube retrieval_manual_evaluation_template.xlsx')
uploaded = files.upload()
print('\nArchivos cargados:', list(uploaded.keys()))

## 3. Leer datos

In [ ]:
input_path = Path('retrieval_manual_evaluation_template.xlsx')

if not input_path.exists():
    raise FileNotFoundError('No se encontró retrieval_manual_evaluation_template.xlsx')

df = pd.read_excel(input_path)

required_cols = {
    'model_alias', 'question_id', 'question', 'categoria', 'rank', 'score',
    'doc_id', 'filename', 'chunk_id', 'tipo_documento', 'page',
    'relevante_manual', 'comentario_evaluador', 'chunk_text'
}

missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Faltan columnas: {missing}')

print('Filas cargadas:', len(df))
print('Modelos:', df['model_alias'].unique())
print('Preguntas:', df['question_id'].nunique())
display(df.head())

## 4. Instrucciones para evaluación manual

Debes revisar los resultados recuperados y marcar la columna `relevante_manual` así:

- `1`: el chunk recuperado sí ayuda a responder la pregunta.
- `0`: el chunk recuperado no ayuda a responder la pregunta.

Para no hacer la revisión demasiado larga, puedes comenzar con una muestra de 10 preguntas.  
Luego, para la versión final de la tesis, completas las 20 preguntas.

## 5. Crear muestra priorizada para evaluación manual

Esta celda crea un archivo más pequeño con 10 preguntas iniciales para revisar manualmente.

In [ ]:
preguntas_prioritarias = [
    'Q001', 'Q002', 'Q003', 'Q004', 'Q005',
    'Q006', 'Q010', 'Q012', 'Q013', 'Q020'
]

df_muestra = df[df['question_id'].isin(preguntas_prioritarias)].copy()

print('Filas para revisión inicial:', len(df_muestra))
print('Preguntas en muestra:', df_muestra['question_id'].nunique())

df_muestra.to_excel('retrieval_manual_evaluation_sample.xlsx', index=False)

display(df_muestra[['model_alias', 'question_id', 'question', 'rank', 'score', 'filename', 'chunk_text', 'relevante_manual']].head(10))

files.download('retrieval_manual_evaluation_sample.xlsx')

## 6. Opción rápida: cargar archivo ya evaluado

Después de descargar `retrieval_manual_evaluation_sample.xlsx`, abre el Excel, llena `relevante_manual` con 1 o 0 y vuelve a subirlo aquí.

Si quieres evaluar todas las preguntas, edita directamente `retrieval_manual_evaluation_template.xlsx`, guárdalo y súbelo de nuevo con el nombre:

`retrieval_manual_evaluation_completed.xlsx`

In [ ]:
print('Cuando tengas el Excel evaluado manualmente, súbelo aquí.')
print('Puede llamarse retrieval_manual_evaluation_completed.xlsx o retrieval_manual_evaluation_sample.xlsx')

uploaded_eval = files.upload()
print('\nArchivos cargados:', list(uploaded_eval.keys()))

## 7. Leer archivo evaluado

In [ ]:
possible_files = [
    'retrieval_manual_evaluation_completed.xlsx',
    'retrieval_manual_evaluation_sample.xlsx'
]

eval_path = None
for p in possible_files:
    if Path(p).exists():
        eval_path = Path(p)
        break

if eval_path is None:
    raise FileNotFoundError('No se encontró archivo evaluado. Sube el Excel con la columna relevante_manual diligenciada.')

eval_df = pd.read_excel(eval_path)

eval_df['relevante_manual'] = pd.to_numeric(eval_df['relevante_manual'], errors='coerce')

sin_evaluar = eval_df['relevante_manual'].isna().sum()
valores_invalidos = eval_df[~eval_df['relevante_manual'].isin([0, 1]) & eval_df['relevante_manual'].notna()]

print('Archivo leído:', eval_path)
print('Filas:', len(eval_df))
print('Filas sin evaluar:', sin_evaluar)
print('Valores inválidos:', len(valores_invalidos))

if len(valores_invalidos) > 0:
    display(valores_invalidos[['model_alias', 'question_id', 'rank', 'relevante_manual']])

display(eval_df.head())

## 8. Filtrar filas evaluadas

Para calcular métricas solo usamos filas donde `relevante_manual` tiene 0 o 1.

In [ ]:
evaluated = eval_df[eval_df['relevante_manual'].isin([0, 1])].copy()
evaluated['relevante_manual'] = evaluated['relevante_manual'].astype(int)

print('Filas evaluadas:', len(evaluated))
print('Preguntas evaluadas:', evaluated['question_id'].nunique())
print('Modelos evaluados:', evaluated['model_alias'].unique())

if evaluated.empty:
    raise ValueError('No hay filas evaluadas. Debes marcar relevante_manual con 1 o 0.')

## 9. Funciones de métricas

In [ ]:
def precision_at_k(group, k):
    topk = group[group['rank'] <= k]
    if len(topk) == 0:
        return 0
    return topk['relevante_manual'].mean()


def hit_at_k(group, k):
    topk = group[group['rank'] <= k]
    if len(topk) == 0:
        return 0
    return int(topk['relevante_manual'].sum() > 0)


def mrr(group):
    ordered = group.sort_values('rank')
    relevant = ordered[ordered['relevante_manual'] == 1]
    if relevant.empty:
        return 0
    first_rank = relevant['rank'].min()
    return 1 / first_rank

## 10. Métricas por pregunta y modelo

In [ ]:
rows = []

for (model_alias, question_id), group in evaluated.groupby(['model_alias', 'question_id']):
    question = group['question'].iloc[0]
    categoria = group['categoria'].iloc[0]

    rows.append({
        'model_alias': model_alias,
        'question_id': question_id,
        'question': question,
        'categoria': categoria,
        'precision_at_1': precision_at_k(group, 1),
        'precision_at_3': precision_at_k(group, 3),
        'precision_at_5': precision_at_k(group, 5),
        'hit_at_1': hit_at_k(group, 1),
        'hit_at_3': hit_at_k(group, 3),
        'hit_at_5': hit_at_k(group, 5),
        'mrr': mrr(group),
        'relevant_retrieved_top5': group[group['rank'] <= 5]['relevante_manual'].sum()
    })

metrics_by_question = pd.DataFrame(rows)
display(metrics_by_question)

## 11. Métricas agregadas por modelo

In [ ]:
metrics_by_model = (
    metrics_by_question.groupby('model_alias')
    .agg(
        questions_evaluated=('question_id', 'nunique'),
        precision_at_1=('precision_at_1', 'mean'),
        precision_at_3=('precision_at_3', 'mean'),
        precision_at_5=('precision_at_5', 'mean'),
        hit_at_1=('hit_at_1', 'mean'),
        hit_at_3=('hit_at_3', 'mean'),
        hit_at_5=('hit_at_5', 'mean'),
        mrr=('mrr', 'mean')
    )
    .reset_index()
)

display(metrics_by_model)

## 12. Métricas por categoría

In [ ]:
metrics_by_category = (
    metrics_by_question.groupby(['model_alias', 'categoria'])
    .agg(
        questions_evaluated=('question_id', 'nunique'),
        precision_at_5=('precision_at_5', 'mean'),
        hit_at_5=('hit_at_5', 'mean'),
        mrr=('mrr', 'mean')
    )
    .reset_index()
)

display(metrics_by_category)

## 13. Gráficos simples de comparación

In [ ]:
ax = metrics_by_model.set_index('model_alias')[['precision_at_1', 'precision_at_3', 'precision_at_5']].plot(kind='bar', figsize=(8, 5))
plt.title('Comparación de Precision@K por modelo')
plt.xlabel('Modelo')
plt.ylabel('Precisión promedio')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title='Métrica')
plt.show()

ax = metrics_by_model.set_index('model_alias')[['hit_at_1', 'hit_at_3', 'hit_at_5']].plot(kind='bar', figsize=(8, 5))
plt.title('Comparación de Hit@K por modelo')
plt.xlabel('Modelo')
plt.ylabel('Hit rate promedio')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title='Métrica')
plt.show()

## 14. Selección preliminar del modelo para RAG baseline

In [ ]:
best_by_p5 = metrics_by_model.sort_values('precision_at_5', ascending=False).iloc[0]
best_by_hit5 = metrics_by_model.sort_values('hit_at_5', ascending=False).iloc[0]

print('Mejor modelo por Precision@5:', best_by_p5['model_alias'])
print('Mejor modelo por Hit@5:', best_by_hit5['model_alias'])

if best_by_p5['model_alias'] == best_by_hit5['model_alias']:
    print('\nRecomendación preliminar: usar', best_by_p5['model_alias'], 'como embedding para el RAG baseline.')
else:
    print('\nLos criterios no coinciden. Revisar manualmente errores y categorías antes de decidir.')

## 15. Guardar resultados

In [ ]:
evaluated.to_excel('retrieval_manual_evaluation_completed.xlsx', index=False)
metrics_by_question.to_csv('retrieval_metrics_by_question.csv', index=False, encoding='utf-8-sig')
metrics_by_model.to_csv('retrieval_metrics_by_model.csv', index=False, encoding='utf-8-sig')
metrics_by_category.to_csv('retrieval_metrics_by_category.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('- retrieval_manual_evaluation_completed.xlsx')
print('- retrieval_metrics_by_question.csv')
print('- retrieval_metrics_by_model.csv')
print('- retrieval_metrics_by_category.csv')

## 16. Descargar resultados

In [ ]:
files.download('retrieval_manual_evaluation_completed.xlsx')
files.download('retrieval_metrics_by_question.csv')
files.download('retrieval_metrics_by_model.csv')
files.download('retrieval_metrics_by_category.csv')